In [0]:
#Validate S3 Paths — Validate S3 URIs such as s3://raw-data/orders/2026/08/orders.csv.
import re
from urllib.parse import urlparse
"""Expected structure:
    s3://<bucket>/<dataset>/<year>/<month>/<filename>"""

def validate_s3_path(s3_uri:str)->dict :
    result={
        "uri":s3_uri,
        "error":[],
        "valid":True,
        "bucket":None,
        "dataset":None,
        "year":None,
        "month":None,
        "filename":None
    }
    # -----------------------------
    # 1. Validate S3 URI scheme
    # -----------------------------
    if not s3_uri.startswith("s3://"):
        result["error"].append("uri must start with s3://")
        result["valid"]=False
        return result 
    # -----------------------------
    # 2. Parse the URI
    # -----------------------------
    parsed=urlparse(s3_uri)
    
    bucket = parsed.netloc
    key=parsed.path.lstrip("/")

    result["bucket"]=bucket
    # -----------------------------
    # 3. Validate bucket name
    # -----------------------------
    match=r"^[a-z0-9][a-z0-9.-]{1-61}[a-z0-9]$"
    if not re.fullmatch(match, bucket):
        result["valid"]=False
        result["error"].append(f"invalid bucket name {bucket}") 
        return result        
    # -----------------------------
    # 4. Split S3 key into folders
    # -----------------------------
    parts=key.split('/')

    if len(parts)!=4:
        result["valid"]=False
        result["error"].append("expected path structure dataset/year/month/file")
        return result
    
    dataset, year, month, filename=parts

    result["dataset"]=dataset
    result["year"]=year
    result["month"]=month
    result["filename"]=filename
    # -----------------------------
    # 5. Validate dataset/folder
    # -----------------------------
    folder_pattern=r"^[a-z]-[a-z0-9-_]*$"
    if not re.fullmatch(folder_pattern, dataset):
        result["valid"]=False
        result["error"].append(f"invalid datase/foldename {dataset}")
    # -----------------------------
    # 6. Validate year partition
    # -----------------------------
    if not re.fullmatch(r"\d{4}", year):
        result["valid"]=False
        result["error"].append(f"invalid year{year}")                       

    # -----------------------------
    # 7. Validate month partition
    # -----------------------------
    if not re.fullmatch(r"(0[1-9]|1[0-2])",month):
        result["valid"]=False
        result["error"].append(f"invalid month {month}")
    # -----------------------------
    # 8. Validate file name
    file_pattern=r"^[a-z][a-z0-9_-]*\.(csv|json|parquet)$"
    if not re.fullmatch(file_pattern, filename):
        result["valid"]=False
        result["error"].append(f"invalid file name{filename}")
    # -----------------------------
    return result
# -------------------------------------------------
# Example paths
# -------------------------------------------------

paths = [
    "s3://raw-data/orders/2026/08/orders.csv",
    "s3://raw-data/orders/2026/13/orders.csv",
    "s3://raw-data/Orders/2026/08/orders.csv",
    "s3://raw-data/orders/26/08/orders.csv",
    "https://raw-data/orders/2026/08/orders.csv",
]


for path in paths:
    validation = validate_s3_path(path)

    print("\nPATH:", path)
    print("VALID:", validation["valid"])

    if validation["error"]:
        for error in validation["error"]:
            print("ERROR:", error)


PATH: s3://raw-data/orders/2026/08/orders.csv
VALID: False
ERROR: invalid bucket name raw-data

PATH: s3://raw-data/orders/2026/13/orders.csv
VALID: False
ERROR: invalid bucket name raw-data

PATH: s3://raw-data/Orders/2026/08/orders.csv
VALID: False
ERROR: invalid bucket name raw-data

PATH: s3://raw-data/orders/26/08/orders.csv
VALID: False
ERROR: invalid bucket name raw-data

PATH: https://raw-data/orders/2026/08/orders.csv
VALID: False
ERROR: uri must start with s3://


In [0]:
#Extract Dates from Filenames — Extract 2026-08-05 from filenames such as orders_2026-08-05.csv.
import re
from datetime import datetime 

def extract_date(filename:str):
    #ext_date="" CGPT-Unnecessary
    date_pattern=r"\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[1-2][0-9]|3[0-1])"
                #r"\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])"-CGPT Version

    match=re.search(date_pattern, filename)

    #CGPT Suggestion
    if not match:
        return None
    
    date_text=match.group()

    # Validate that it is a real calendar date
    try:
        valid_date=datetime.strptime(date_text, "%Y-%m-%d")
        return valid_date.date()
    except ValueError:
        return None

date_val=extract_date("orders_2026-01-30.csv")
print(date_val)

2026-01-30


In [0]:
#Parse Log Records — Extract timestamp, log level, pipeline name, and error message from ETL application logs. 
#1.Download sample error, warning & success logs
#2.log level- if its warning/error/success
#3. ENHANCEMENT- when error- move errored records to DLQ
#4.ENHANCEMENT-Read from airflow log file and store email results to team with level of log & job name as header
#5.ENHANCEMENT-Modify the code for multi line log file

#sample log- 2026-08-08 14:32:10 ERROR pipeline=orders_etl status=FAILED message=Unable to connect to Snowflake

from datetime import datetime
import re 
def parse_etl_log(etlname:str)->dict:
    result={
        "timestamp":None,
        "log_level":None,
        "pipeline_name":None,
        "error_message":None,
        "warning_message":None,
        "Info":None,
        "status":None
    } 
    #timestamp
    #^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$
    tmp_pattern=(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")
    tmp_match=re.search(tmp_pattern,etlname )
    if tmp_match:
        result["timestamp"]=tmp_match.group()
    
    #log level#
    log_pattern=r"ERROR|WARN|INFO|ERROR|CRITICAL"
    level_match=re.search(log_pattern,etlname)
    if level_match:
        result["log_level"]=level_match.group()

    #pipeline name
    job_title=r"pipeline=([a-zA-Z0-9_-]+)"
    job_title_match=re.search(job_title,etlname)
    if job_title_match:
        result["pipeline_name"] =job_title_match.group(1)

    #error message
    message=r"message=([a-zA-Z_-](.+))"
    message_match=re.search(message,etlname)
    if message_match:
        result["error_message"]=message_match.group(1)
    
    #status
    stat=r"FAILED|SUCCESS"
    stat_match=re.search(stat,etlname)
    if stat_match:
        result["status"]=stat_match.group()

    return result

log = (
    "2026-08-08 14:32:10 ERROR "
    "pipeline=orders_etl "
    "status=FAILED "
    "message=Unable to connect to Snowflake"
)

parsed_log = parse_etl_log(log)
print(parsed_log)

{'timestamp': None, 'log_level': 'ERROR', 'pipeline_name': 'orders_etl', 'error_message': 'Unable to connect to Snowflake', 'warning_message': None, 'Info': None, 'status': 'FAILED'}


In [0]:
#Validate Email Fields — Identify invalid email addresses in incoming customer data.
#test data=> przirape7@gmail.com    && abc.com

import re
def email_validation(cust_data:str)->bool:
    email_pattern= r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    
    if not cust_data :
        return False
    
    cust_data=cust_data.strip()

    return bool(re.fullmatch(email_pattern, cust_data))

cust_data="john.doe@email.com"
print(cust_data ,email_validation(cust_data))

john.doe@email.com True


In [0]:
#Extract Table Names from SQL — Extract table names appearing after FROM and JOIN in SQL queries.
import re
def tablename_parsing(sql_query:str)->list[str]:
    pattern=r"\b(?:FROM|JOIN)\s+([a-zA-Z_][a-zA-Z_.]+)"

    match=re.findall(pattern,sql_query, flags=re.IGNORECASE)

    return list(dict.fromkeys(match))

sql_query='''SELECT 
    customers.customer_id,
    customers.customer_name,
    orders.order_id,
    orders.order_date,
    orders.total_amount
FROM maw.customers
INNER JOIN raw.orders 
    ON customers.customer_id = orders.customer_id
WHERE orders.order_date >= '2026-01-01'
ORDER BY orders.order_date DESC;'''

tablename_parsing(sql_query)


['maw.customers', 'raw.orders']

In [0]:
#Clean Column Names — Convert column names such as Customer ID, Order-Amount($), and Order Date into standardized snake_case names.

#Snake case is a naming style where spaces are replaced with underscores (_) and all letters are written in lowercase. 

import re
def snake_case_parse(col_name:str)->str:
    col_space_strip= col_name.strip()
    col_lower_case=col_space_strip.lower()
    col_pattern=re.sub(r"[^a-z0-9]+","_", col_lower_case)
    col_underscore_strip=col_pattern.strip("_")
    return col_underscore_strip

def unique_col_check(cols:list[str])->list[str]:
    cleaned_cols=[snake_case_parse(col) 
                  for col in cols]
    
    if len(cleaned_cols)!=len(set(cleaned_cols)):
        raise ValueError(f"Duplicate columns after cleaning: {cleaned_cols}")
    return cleaned_cols

columns = [
    "Customer ID",
    "Order-Amount($)",
    "Order Date",
]

print(unique_col_check(columns))

['customer_id', 'order_amount', 'order_date']


In [0]:
#Validate Data File Names — Accept filenames following a convention such as orders_20260805_143000.csv and reject invalid ones.

import re
from datetime import datetime
def file_name_validation(filename:str)->dict:
    result={
        "file_name":None,
        "valid":False
    }
    if len(filename) <12:
        result["filename"]   =filename
        return result
    
    date_pattern=r"\d{8}"
    dates=re.search(date_pattern, filename)

    if dates:
        date_str=dates.group()
    try:
        datetime.strptime(date_str, '%Y%m%d')
    except ValueError:
        print("Invalid Date")
        #result["filename"]=filename
        #return result            

    pattern=r"[a-z]+_\d{8}_\d{6}\.(?:csv|json|parquet$)"
    match=re.fullmatch(pattern,filename,re.IGNORECASE)
    if match:
        result["valid"]=True
    result["file_name"]=filename 
    return result

filename="orders_20260905_143000.csv"
print(file_name_validation(filename))   


{'file_name': 'orders_20260905_143000.csv', 'valid': True}


In [0]:
#COMPLETE ABOVE PGM FIRST